In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import json
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [129]:
data = json.load(open("data/pokemon-sets.json"))

In [130]:
rows = []
cardSkipCount = 0
entrySkipCount = 0

for seriesId, series in data.items():
    for card in series["cards"]:
        if len(card["priceEvolutionCM"]) == 0 or len(card["priceEvolutionTCG"]) == 0:
            cardSkipCount += 1
            print(f"Skipping card {card['name']} from series {seriesId} because it has no initial price in cardmarket data")
            continue

        currentPriceCM = None
        i = 0
        while i < len(card["priceEvolutionCM"]):
            if "price" in card["priceEvolutionCM"][i]:
                currentPriceCM = card["priceEvolutionCM"][i]["price"]
                break

            i +=1

        if currentPriceCM is None:
            cardSkipCount += 1
            print(f"Skipping card {card['name']} from series {seriesId} because it has no initial price in cardmarket data")
            continue


        currentPriceTCG = None
        j = 0
        while j < len(card["priceEvolutionTCG"]):
            if "price" in card["priceEvolutionTCG"][j]:
                currentPriceTCG = card["priceEvolutionTCG"][j]["price"]
                break

            j += 1

        if currentPriceTCG is None:
            cardSkipCount += 1
            print(f"Skipping card {card['name']} from series {seriesId} because it has no initial price in tcgplayer data")
            continue

        t = 0
        for cardmarketdata, tcgplayerdata in zip(card["priceEvolutionCM"], card["priceEvolutionTCG"]):
            currentPriceCM = cardmarketdata["price"] if "price" in cardmarketdata else currentPriceCM
            currentPriceTCG = tcgplayerdata["price"] if "price" in tcgplayerdata else currentPriceTCG
            rows.append({
                "t": t,
                "seriesId": seriesId,
                "commercial_name": series["commercialName"],
                "hype_level": series["hypeLevel"],
                "age_level": series["ageLevel"],

                "card_id": card["id"],
                "name": card["name"],
                "number": card["number"],
                "rarity": card["rarity"],
                "pokedex_id": card["pokedexId"],
                "image": card["image"],

                "current_offer": sum(list(map(lambda x: x["number"], card["offer"]))),
                "current_demand": sum(list(map(lambda x: x["number"], card["demand"]))),

                "cardmarket_price": currentPriceCM,
                "tcgplayer_price": currentPriceTCG
            })

            t += 1

df = pd.DataFrame(rows)
print(f"Dataframe shape: {df.shape} with {cardSkipCount} skipped cards and {entrySkipCount} skipped entries")
df.head()

Skipping card Salarsen from series PFL because it has no initial price in cardmarket data
Skipping card Reshiram de N from series JTG because it has no initial price in cardmarket data
Dataframe shape: (173342, 15) with 2 skipped cards and 0 skipped entries


,t,seriesId,commercial_name,hype_level,age_level,card_id,name,number,rarity,pokedex_id,image,current_offer,current_demand,cardmarket_price,tcgplayer_price
0,0,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,8.0,4.82
1,1,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,7.9,4.98
2,2,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,7.9,4.47
3,3,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,7.9,5.23
4,4,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,7.9,4.46


In [131]:
T = 60
X = 14

price_T = df[df['t'] == T].set_index('card_id')['cardmarket_price']
price_TX = df[df['t'] == (T + X)].set_index('card_id')['cardmarket_price']

labels = ((price_TX - price_T) > 0).astype(int).rename('target').reset_index()

In [132]:
labels["target"].value_counts()

target
1    771
0    355
Name: count, dtype: int64

In [133]:
df_hist = df[df['t'] <= T].copy()
df_hist = df_hist.sort_values(['card_id', 't'])
df_hist['daily_return'] = df_hist.groupby('card_id')['cardmarket_price'].pct_change()

In [134]:
def get_price_at_t(group, target_t):
    past = group[group['t'] <= target_t]
    if past.empty:
        return np.nan
    
    return past.iloc[-1]['cardmarket_price']


def extract_features(group):
    group = group.sort_values('t')
    latest = group.iloc[-1]

    T = latest['t']

    price_t = latest['cardmarket_price']
    tcg_price_t = latest['tcgplayer_price']

    price_t_7 = get_price_at_t(group, T - 7)
    price_t_14 = get_price_at_t(group, T - 14)
    price_t_30 = get_price_at_t(group, T - 30)

    f = {
        'card_id': latest['card_id'],
        'seriesId': latest['seriesId'],
        'rarity': latest['rarity'],
        'hype_level': latest['hype_level'],
        'age_level': latest['age_level'],

        # Supply / Demand
        'current_offer': latest['current_offer'],
        'current_demand': latest['current_demand'],
        'demand_to_offer_ratio': (
            latest['current_demand'] /
            (latest['current_offer'] + 1e-5)
        ),

        # Price
        'price_T': price_t,
        'cross_market_ratio': (
            price_t / (tcg_price_t + 1e-5)
        ),

        # Momentum
        'return_7d': (
            (price_t - price_t_7) / price_t_7
            if pd.notna(price_t_7) and price_t_7 != 0
            else np.nan
        ),

        'return_14d': (
            (price_t - price_t_14) / price_t_14
            if pd.notna(price_t_14) and price_t_14 != 0
            else np.nan
        ),

        'return_30d': (
            (price_t - price_t_30) / price_t_30
            if pd.notna(price_t_30) and price_t_30 != 0
            else np.nan
        ),

        # Volatility
        'volatility_14d': group.tail(14)['daily_return'].std(),
        'volatility_30d': group.tail(30)['daily_return'].std(),

        # Moving averages
        'sma_7_ratio': (
            price_t / group.tail(7)['cardmarket_price'].mean()
        ),

        'sma_30_ratio': (
            price_t / group.tail(30)['cardmarket_price'].mean()
        ),
    }

    return pd.Series(f)

In [135]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

features_df = df_hist.groupby('card_id').apply(extract_features).reset_index(drop=True)

dataset = pd.merge(features_df, labels, on='card_id')
t
cat_cols = ['seriesId', 'rarity']
for col in cat_cols:
    dataset[col] = dataset[col].astype('category')

feature_cols = [c for c in dataset.columns if c not in ['card_id', 'target']]
X_data = dataset[feature_cols]
y_data = dataset['target']
groups = dataset['card_id']

gkf = GroupKFold(n_splits=5)

oof_preds = np.zeros(len(dataset))
feature_importances = pd.Series(0.0, index=feature_cols)

num_neg = (y_data == 0).sum()
num_pos = (y_data == 1).sum()

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.02,
    'num_leaves': 10,              # Conservative tree size
    'min_child_samples': 30,       # Require strong evidence for splits
    'colsample_bytree': 0.8,       # Feature subsampling to reduce variance
    'subsample': 0.8,              # Data subsampling
    'verbose': -1
}
print("Starting GroupKFold Cross-Validation...\n")

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_data, y_data, groups=groups)):
    X_train, y_train = X_data.iloc[train_idx], y_data.iloc[train_idx]
    X_val, y_val = X_data.iloc[val_idx], y_data.iloc[val_idx]
    
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=300,
        valid_sets=[train_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    
    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    oof_preds[val_idx] = val_preds
    
    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"Fold {fold + 1} AUC: {fold_auc:.4f}")
    
    feature_importances += model.feature_importance(importance_type='gain') / gkf.n_splits

/var/folders/63/z6gq_ry925b6ttrn52rwrwp80000gn/T/ipykernel_66539/3002450457.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features_df = df_hist.groupby('card_id').apply(extract_features).reset_index(drop=True)


Starting GroupKFold Cross-Validation...

Fold 1 AUC: 0.7808
Fold 2 AUC: 0.6790
Fold 3 AUC: 0.7594
Fold 4 AUC: 0.7722
Fold 5 AUC: 0.8152


In [136]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, classification_report

# Find the threshold that guarantees at least 85% Precision on the "Up" class (Class 1)
target_precision = 0.85
thresholds = np.linspace(0.5, 0.95, 91)

selected_thresh = 0.5
achieved_precision = 0.0

for t in thresholds:
    preds_binary = (oof_preds >= t).astype(int)
    # Class 1 Precision
    prec = precision_score(y_data, preds_binary, pos_label=1, zero_division=0)
    
    if prec >= target_precision:
        selected_thresh = t
        achieved_precision = prec
        break

print(f"Optimal High-Confidence Threshold: {selected_thresh:.3f}")
print(f"Achieved Precision for 'Up': {achieved_precision:.2%}")

# Evaluate overall impact on predictions
oof_high_conf = (oof_preds >= selected_thresh).astype(int)
print(classification_report(y_data, oof_high_conf, target_names=['Down/Flat/Ignore', 'High-Confidence Up']))

Optimal High-Confidence Threshold: 0.780
Achieved Precision for 'Up': 85.98%
                    precision    recall  f1-score   support

  Down/Flat/Ignore       0.39      0.87      0.54       355
High-Confidence Up       0.86      0.37      0.51       771

          accuracy                           0.52      1126
         macro avg       0.62      0.62      0.52      1126
      weighted avg       0.71      0.52      0.52      1126

